# Game of Thoughts — Final Analysis

## Research Question
Can large language models understand existing game rules and generate new games with logically consistent rule systems?

This notebook presents the final analysis for the project and compares two models:

- Gemini
- Meta Llama 3 via Groq

The notebook focuses on:
1. Rule understanding
2. Move prediction
3. Multi-turn simulation
4. Rule error detection
5. Game generation


## Experimental Pipeline

This project evaluates LLM reasoning in a rule-based environment using Tic-Tac-Toe.

Experiments:
1. **Rule Understanding** — explain clean, raw, and broken rules
2. **Move Prediction** — suggest a valid move from a board state
3. **Multi-Turn Simulation** — maintain coherent play across turns
4. **Rule Error Detection** — detect inconsistencies in broken rules
5. **Game Generation** — generate a new board game with consistent rules


In [8]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "results").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

print("Project root:", ROOT)


Project root: /workspaces/game-of-thoughts


## Load Rule Understanding Results

In [9]:
gemini_rules = pd.read_csv(ROOT / "results" / "tables" / "rule_understanding_summary_gemini.csv")
groq_rules = pd.read_csv(ROOT / "results" / "tables" / "rule_understanding_summary_groq.csv")

print("Gemini")
display(gemini_rules)

print("Groq / Llama")
display(groq_rules)


Gemini


,condition,rule_completeness_score,invented_rules,missed_constraints,notes
0,clean,0.86,0,1,NaN
1,raw,1.00,0,0,NaN
2,broken,0.71,0,2,NaN


Groq / Llama


,condition,rule_completeness_score,invented_rules,missed_constraints,notes
0,clean,0.86,0,1,NaN
1,raw,0.86,0,1,NaN
2,broken,0.71,0,2,NaN


The rule understanding experiment evaluates how well each model preserves rule constraints under three conditions:

- clean rules
- raw noisy rulebook
- broken rules

Metrics:
- rule completeness score
- invented rules
- missed constraints


## Load Move Prediction Results

In [10]:
gemini_moves = pd.read_csv(ROOT / "results" / "move_prediction_gemini.csv")
groq_moves = pd.read_csv(ROOT / "results" / "move_prediction_groq.csv")

gemini_valid = gemini_moves["is_valid"].mean()
groq_valid = groq_moves["is_valid"].mean()

gemini_optimal = gemini_moves["is_optimal"].mean()
groq_optimal = groq_moves["is_optimal"].mean()

pd.DataFrame({
    "model": ["Gemini", "Groq / Llama"],
    "valid_move_rate": [gemini_valid, groq_valid],
    "optimal_move_rate": [gemini_optimal, groq_optimal],
})


,model,valid_move_rate,optimal_move_rate
0,Gemini,0.95,0.1
1,Groq / Llama,0.90,0.1


Move prediction tests whether the model can apply rules to concrete game states.

Metrics:
- **valid move rate** = proportion of legal moves
- **optimal move rate** = proportion of moves that immediately win when such a move is selected


## Load Multi-Turn Simulation Results

In [11]:
gemini_multi = pd.read_csv(ROOT / "results" / "tables" / "multi_turn_summary_gemini.csv")
groq_multi = pd.read_csv(ROOT / "results" / "tables" / "multi_turn_summary_groq.csv")

print("Gemini")
display(gemini_multi)

print("Groq / Llama")
display(groq_multi)


Gemini


,case,total_turns,valid_turns,coherence_rate,winner
0,midgame_1,2,2,1.0,X
1,midgame_2,4,4,1.0,NaN
2,midgame_3,3,3,1.0,X
3,midgame_4,1,0,0.0,NaN
4,midgame_5,4,4,1.0,NaN
5,midgame_6,1,1,1.0,X
6,midgame_7,3,3,1.0,O
7,midgame_8,3,3,1.0,O
8,midgame_9,1,0,0.0,NaN
9,midgame_10,2,1,0.5,NaN


Groq / Llama


,case,total_turns,valid_turns,coherence_rate,winner
0,midgame_1,1,0,0.00,NaN
1,midgame_2,4,4,1.00,NaN
2,midgame_3,2,1,0.50,NaN
3,midgame_4,1,0,0.00,NaN
4,midgame_5,1,0,0.00,NaN
5,midgame_6,4,4,1.00,NaN
6,midgame_7,4,4,1.00,X
7,midgame_8,1,0,0.00,NaN
8,midgame_9,4,3,0.75,NaN
9,midgame_10,4,3,0.75,NaN


In [12]:
gemini_coherence = gemini_multi["valid_turns"].sum() / gemini_multi["total_turns"].sum()
groq_coherence = groq_multi["valid_turns"].sum() / groq_multi["total_turns"].sum()

pd.DataFrame({
    "model": ["Gemini", "Groq / Llama"],
    "coherence_rate": [gemini_coherence, groq_coherence],
    "total_turns": [gemini_multi["total_turns"].sum(), groq_multi["total_turns"].sum()],
    "valid_turns": [gemini_multi["valid_turns"].sum(), groq_multi["valid_turns"].sum()],
})


,model,coherence_rate,total_turns,valid_turns
0,Gemini,0.875000,24,21
1,Groq / Llama,0.730769,26,19


The multi-turn simulation evaluates whether model decisions remain coherent across turns.

Metric:
- **coherence rate** = valid turns / total turns


## Rule Error Detection Outputs

In [13]:
gemini_error = (ROOT / "results" / "rule_error_detection_gemini.txt").read_text(encoding="utf-8")
groq_error = (ROOT / "results" / "rule_error_detection_groq.txt").read_text(encoding="utf-8")

print("Gemini error detection output:")
print(gemini_error[:2000])

print("\n" + "="*80 + "\n")

print("Groq / Llama error detection output:")
print(groq_error[:2000])


Gemini error detection output:
BROKEN RULES

Tic-Tac-Toe is played on a 3x3 grid.
Two players take turns placing X and O.
A player can place a mark anywhere.
The first player to place three marks in a row wins.
The game ends when the board is full.


PROMPT


You are given the rules of a board game.

Game rules:
Tic-Tac-Toe is played on a 3x3 grid.
Two players take turns placing X and O.
A player can place a mark anywhere.
The first player to place three marks in a row wins.
The game ends when the board is full.

Task:
1. Identify missing or ambiguous rules.
2. Explain why they are problematic.
3. Suggest a corrected version of the rules.



MODEL RESPONSE

To improve the clarity and completeness of the provided Tic-Tac-Toe rules, I have identified several critical gaps.

### 1. Missing or Ambiguous Rules

*   **Starting Player:** The rules do not specify who goes first (e.g., "Player X goes first").
*   **Winning Conditions:** The rules state "three marks in a row," but do not clarify

This experiment evaluates whether the models can identify:

- missing rules
- ambiguous rules
- contradictory rules
- possible corrections


## Game Generation Outputs

In [14]:
gemini_game = (ROOT / "results" / "game_generation_gemini.txt").read_text(encoding="utf-8")
groq_game = (ROOT / "results" / "game_generation_groq.txt").read_text(encoding="utf-8")

print("Gemini generated game:")
print(gemini_game[:2500])

print("\n" + "="*80 + "\n")

print("Groq / Llama generated game:")
print(groq_game[:2500])


Gemini generated game:
PROMPT

Design a simple two-player board game.

Theme: a simple two-player strategy board game about collecting treasure

Provide:
1. Game name
2. Objective
3. Setup
4. Turn structure
5. Rules
6. Winning condition

Ensure the game rules are logically consistent.



MODEL RESPONSE
### Game Name: **Golden Gulch**

**Theme:** Two treasure hunters explore a shifting, ancient canyon, trying to secure the most gold before the canyon collapses.

---

### 1. Objective
Collect more gold coins than your opponent. The game ends when all gold coins have been collected from the board.

### 2. Setup
*   **The Board:** Use a 5x5 grid (25 squares total).
*   **The Tokens:** Each player chooses a color (Player A: Blue, Player B: Red). Each player has **two** explorer pawns.
*   **The Treasure:** Place 10 gold tokens randomly on the 25 squares. The remaining 15 squares are empty.
*   **Starting Positions:** Players place their two pawns on the two corner squares closest to them (e

The generated games are evaluated using simple qualitative criteria:

- structure completeness
- logical consistency
- playability
- rule clarity


## Model Comparison Table

In [15]:
comparison = pd.DataFrame({
    "model": ["Gemini", "Groq / Llama"],
    "valid_move_rate": [gemini_valid, groq_valid],
    "optimal_move_rate": [gemini_optimal, groq_optimal],
    "coherence_rate": [gemini_coherence, groq_coherence],
    "avg_rule_completeness": [
        gemini_rules["rule_completeness_score"].mean(),
        groq_rules["rule_completeness_score"].mean()
    ],
})

comparison


,model,valid_move_rate,optimal_move_rate,coherence_rate,avg_rule_completeness
0,Gemini,0.95,0.1,0.875000,0.856667
1,Groq / Llama,0.90,0.1,0.730769,0.810000


## Interpretation

Suggested interpretation points:

- Both models perform well on **rule adherence**
- Gemini appears stronger on **valid move accuracy**
- Groq / Llama may produce more **creative** generations, but sometimes with less precision
- Multi-turn coherence shows whether the models remain consistent beyond one-step responses


## Conclusion

This project evaluates how LLMs balance creativity and logical constraint in rule-based environments.

Main findings:
- Both models can understand structured rules
- Both models can generate mostly valid moves
- Multi-turn simulation shows whether rule consistency persists over time
- Generated games reveal differences in how models trade off structure and creativity

Overall, the experiments show that LLMs can internalize and extend structured rule systems, but performance differs depending on the task and model.
